# Safira | Pipeline do modelo final 

**Fluxo:** artefato confiavel -> avaliacao -> explicacao -> entrada operacional -> score -> fila diaria. O notebook carrega o Gradient Boosting calibrado e seu preprocessador; nao treina, nao recalibra e nao escolhe hiperparametros.

Execute todas as celulas antes da gravacao e apresente as cinco cenas abaixo. O modo de video usa exclusivamente dados e artefato sinteticos. Numeros sinteticos nao sao resultados da Azul. Para resultados historicos, consulte a Secao 4.5 de `documents/documentacao.md` e `comparacao_modelos.ipynb`.

## Preparacao local
Na raiz do repositorio, no mesmo ambiente Python usado para gerar o Joblib:
```bash
python -m pip install -r requirements-treinamento.txt jupyterlab ipykernel matplotlib
python scripts/gerar_artefato_modelo.py --sintetico --modelo artifacts/modelo-sintetico-v1.joblib --manifesto artifacts/manifesto-sintetico-v1.json --versao-modelo sintetico-v1 --versao-fontes artificial
python -m jupyter lab notebooks/pipeline_modelo_final.ipynb
```
O gerador recusa sobrescrita: se o par ja existir, reutilize-o ou gere uma nova versao com outros nomes. Selecione o kernel do mesmo ambiente. Hash nao autentica fornecedor: carregue apenas arquivos de origem confiavel. Outputs reais, binarios, CSVs e notebooks executados permanecem locais; nunca publique registros reais no video.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/artefato_modelo.py').exists())
for pasta in ('src', 'scripts'):
    sys.path.insert(0, str(RAIZ / pasta))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import (RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
                             brier_score_loss, precision_score)
from sklearn.calibration import CalibrationDisplay
from sklearn.inspection import permutation_importance
from threadpoolctl import threadpool_limits
from artefato_modelo import carregar_artefato, sha256
from avaliacao import avaliar
from contrato_inferencia import COLUNAS_MODELO, COLUNAS_PASSAGEM, validar_entrada
from dados_sinteticos_operacionais import criar_base_sintetica, criar_entrada_sintetica
from features import adicionar_historico
from split import dividir, conferir
from preprocessamento_nps import selecionar_features_score_pos_viagem, validar_contrato_dados_score_pos_viagem

MODO_VIDEO = os.environ.get('SAFIRA_MODO_VIDEO', '1') == '1'
USAR_BASE_SINTETICA = os.environ.get('SAFIRA_BASE_SINTETICA', '1') == '1'
CAMINHO_MODELO = Path(os.environ.get('SAFIRA_MODELO', str(RAIZ / 'artifacts/modelo-sintetico-v1.joblib')))
CAMINHO_MANIFESTO = Path(os.environ.get('SAFIRA_MANIFESTO', str(RAIZ / 'artifacts/manifesto-sintetico-v1.json')))
CAMINHO_BASE_REAL = RAIZ / 'data/processed/base_analitica.parquet'
LIMIAR_DIAGNOSTICO = 0.5  # Apenas matriz de confusao; nao controla a fila.
CAPACIDADE_DIARIA = 50
T_SCORE = '2026-10-06T12:00:00Z'
FUSO = 'America/Sao_Paulo'
DIA_DEMO = '2026-10-06'
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.2})

if MODO_VIDEO and not USAR_BASE_SINTETICA:
    raise ValueError('Video publico permite somente demonstracao sintetica.')
modelo, manifesto = carregar_artefato(CAMINHO_MODELO, CAMINHO_MANIFESTO)
if manifesto['dados_sinteticos'] != USAR_BASE_SINTETICA:
    raise ValueError('A origem do artefato deve corresponder ao modo da base de avaliacao.')
TIPO_RESULTADO = 'DEMONSTRACAO SINTETICA' if USAR_BASE_SINTETICA else 'AVALIACAO REAL LOCAL - NAO PUBLICAR OUTPUTS'
display(Markdown(f'**{TIPO_RESULTADO}** | Modelo: `{manifesto["versao_modelo"]}` | Aprovado para producao: `{manifesto["aprovado_producao"]}`'))


## Cena 1 | Problema e solucao 
**Tela:** README do projeto e este resumo.




## Cena 2 | Resultado no teste 
**Tela:** tres indicadores e curvas ROC/Precisao-Recall.





In [ ]:
if USAR_BASE_SINTETICA:
    base = criar_base_sintetica()
else:
    if sha256(CAMINHO_BASE_REAL) != manifesto['fingerprint_base']:
        raise ValueError('A base real difere da origem registrada no artefato; avaliacao interrompida.')
    base = pd.read_parquet(CAMINHO_BASE_REAL)

base_historica = adicionar_historico(base)
validar_contrato_dados_score_pos_viagem(base_historica, exigir_alvo=True)
particoes, metadados_split = dividir(base_historica, corte_validacao=manifesto['cortes']['validacao'], corte_teste=manifesto['cortes']['teste'], sem_data='treino')
conferir(particoes, metadados=metadados_split)
features_v1, _, _ = selecionar_features_score_pos_viagem(base_historica)
x_historico = pd.concat([features_v1, base_historica.loc[features_v1.index, list(COLUNAS_MODELO[11:])]], axis=1)
x_teste = x_historico.loc[particoes['teste'].index, list(COLUNAS_MODELO)]
y_teste = particoes['teste']['DETRATOR']
if set(y_teste.unique()) != {0, 1}:
    raise ValueError('Avaliacao exige teste nao vazio com as duas classes.')
indice_positivo = list(modelo.classes_).index(1)
with threadpool_limits(limits=1):
    score_teste = modelo.predict_proba(x_teste)[:, indice_positivo]
pred_teste = (score_teste >= LIMIAR_DIAGNOSTICO).astype(int)
metricas = avaliar(y_teste, pred_teste, score_teste)
metricas['Brier'] = brier_score_loss(y_teste, score_teste)
metricas['Precisao no limiar'] = precision_score(y_teste, pred_teste, zero_division=0)
metricas['Prevalencia'] = float(y_teste.mean())
display(Markdown(f'### {TIPO_RESULTADO}: {len(y_teste)} jornadas no teste'))
display(pd.Series({nome: metricas[nome] for nome in ('Precisão Média', 'ROC-AUC', 'Brier')}, name='Resultado').to_frame().round(4))
display(pd.Series(metricas, name='Diagnostico completo').to_frame().round(4))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
RocCurveDisplay.from_predictions(y_teste, score_teste, ax=axes[0, 0], name='GB calibrado')
axes[0, 0].plot([0, 1], [0, 1], '--', color='gray', label='Aleatorio')
axes[0, 0].set_title('ROC | discriminacao')
PrecisionRecallDisplay.from_predictions(y_teste, score_teste, ax=axes[0, 1], name='GB calibrado')
axes[0, 1].axhline(y_teste.mean(), linestyle='--', color='gray', label='Prevalencia')
axes[0, 1].legend()
axes[0, 1].set_title('Precisao-Recall | classe Detrator')
CalibrationDisplay.from_predictions(y_teste, score_teste, n_bins=8, strategy='quantile', ax=axes[1, 0], name='GB calibrado')
axes[1, 0].set_title('Calibracao | risco estimado e observado')
ConfusionMatrixDisplay.from_predictions(y_teste, pred_teste, display_labels=['Nao Detrator', 'Detrator'], ax=axes[1, 1], colorbar=False)
axes[1, 1].grid(False)
axes[1, 1].set_title(f'Matriz no limiar {LIMIAR_DIAGNOSTICO} | nao e a fila diaria')
fig.suptitle(TIPO_RESULTADO)
fig.tight_layout()
plt.show()


## Cena 3 | Fatores associados ao score 
**Tela:** importancia por permutacao das features brutas na validacao, sem mudar o modelo.



In [ ]:
validacao = particoes['validacao']
if len(validacao) > 1000:
    validacao = validacao.groupby('DETRATOR', group_keys=False).sample(frac=1000 / len(validacao), random_state=42)
x_explicacao = x_historico.loc[validacao.index, list(COLUNAS_MODELO)]
y_explicacao = validacao['DETRATOR']
if set(y_explicacao.unique()) != {0, 1}:
    raise ValueError('Importancia exige validacao com as duas classes.')
with threadpool_limits(limits=1):
    importancia = permutation_importance(modelo, x_explicacao, y_explicacao, scoring='average_precision', n_repeats=3, random_state=42, n_jobs=1)
fatores = pd.DataFrame({'feature': COLUNAS_MODELO, 'queda_PM': importancia.importances_mean, 'desvio': importancia.importances_std})
display(fatores.sort_values('queda_PM', ascending=False).reset_index(drop=True).style.format({'queda_PM': '{:.6f}', 'desvio': '{:.6f}'}))
if USAR_BASE_SINTETICA:
    display(Markdown('**Limite da demonstracao:** o alvo artificial foi construido a partir do atraso; ATRASO_CHEGADA repete ESTATISTICA_ATRASOSAIDA e varias features sao constantes. Uma unica barra relevante e possivel e nao representa os fatores reais da Azul. Importancia zero nesta amostra nao prova irrelevancia geral.'))
destaques = fatores.sort_values('queda_PM', ascending=False).head(8).sort_values('queda_PM')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(destaques['feature'], destaques['queda_PM'], xerr=destaques['desvio'], color='#176b87')
ax.axvline(0, color='gray', linewidth=1)
ax.set_xlabel('Queda de Precisao Media ao embaralhar | maior = maior dependencia')
ax.set_title(f'Fatores do modelo | {TIPO_RESULTADO}')
fig.tight_layout()
plt.show()


## Cena 4 | Uso na pratica: fila de um dia 
**Tela:** resumo da execucao e primeiras posicoes da fila.


In [ ]:
def pontuar_fila(entrada, *, capacidade, t_score, fuso):
    if type(capacidade) is not int or capacidade < 0:
        raise ValueError('Capacidade diaria deve ser inteiro nao negativo.')
    normalizada, correcoes = validar_entrada(entrada, t_score=t_score, fuso_operacional=fuso)
    saida = normalizada.loc[:, COLUNAS_PASSAGEM + ['DIA_OPERACIONAL']].copy()
    with threadpool_limits(limits=1):
        scores = modelo.predict_proba(normalizada.loc[:, list(COLUNAS_MODELO)])[:, indice_positivo] if len(normalizada) else np.array([])
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError('Modelo produziu probabilidades invalidas; fila interrompida.')
    saida['PROBABILIDADE_DETRACAO'] = scores
    saida = saida.sort_values(['DIA_OPERACIONAL', 'PROBABILIDADE_DETRACAO', 'ID_CLIENTE', 'ID_JORNADA'], ascending=[True, False, True, True], kind='stable').reset_index(drop=True)
    representante = ~saida.duplicated(['DIA_OPERACIONAL', 'ID_CLIENTE'])
    saida['POSICAO_FILA'] = pd.Series(pd.NA, index=saida.index, dtype='Int64')
    saida.loc[representante, 'POSICAO_FILA'] = saida.loc[representante].groupby('DIA_OPERACIONAL').cumcount() + 1
    saida['PRIORIZADO'] = (saida['POSICAO_FILA'] <= capacidade).fillna(False).astype(bool)
    saida['FAIXA_RISCO'] = 'NAO_VALIDADA'
    saida['VERSAO_MODELO'] = manifesto['versao_modelo']
    saida['VERSAO_CONTRATO'] = manifesto['versao_contrato']
    saida['VERSAO_POLITICA_FILA'] = manifesto['versao_politica_fila']
    saida['VERSAO_FONTES'] = 'fotografia-artificial-video-v1'
    saida['FUSO_OPERACIONAL'] = fuso
    saida['T_SCORE'] = t_score
    saida['ID_EXECUCAO'] = 'demonstracao-artificial-video-v1'
    return saida, correcoes

entrada_demo = pd.concat([criar_entrada_sintetica()] * 10, ignore_index=True)
entrada_demo['ID_CLIENTE'] = [f'CLIENTE-ARTIFICIAL-{i // 2:03}' for i in range(len(entrada_demo))]
entrada_demo['ID_JORNADA'] = [f'JORNADA-ARTIFICIAL-{i:03}' for i in range(len(entrada_demo))]
entrada_demo['T_EVENTO_ELEGIBILIDADE'] = '2026-10-06T10:00:00Z'
entrada_demo['T_DISPONIBILIDADE_FEATURES'] = '2026-10-06T11:00:00Z'
fila, correcoes = pontuar_fila(entrada_demo, capacidade=CAPACIDADE_DIARIA, t_score=T_SCORE, fuso=FUSO)
fila_dia = fila.loc[fila['DIA_OPERACIONAL'] == DIA_DEMO]
contatos = fila_dia.loc[fila_dia['PRIORIZADO']]
display(Markdown('### FILA SIMULADA | dados artificiais, sem contatos reais'))
display(pd.Series({'Jornadas pontuadas': len(fila_dia), 'Clientes elegiveis': fila_dia['ID_CLIENTE'].nunique(), 'Contatos sugeridos': len(contatos), 'Capacidade diaria': CAPACIDADE_DIARIA}, name=DIA_DEMO).to_frame())
display(contatos[['POSICAO_FILA', 'ID_CLIENTE', 'ID_JORNADA', 'PROBABILIDADE_DETRACAO', 'PRIORIZADO']].head(10).style.format({'PROBABILIDADE_DETRACAO': '{:.2%}'}))
fig, ax = plt.subplots(figsize=(9, 4))
top10 = contatos.head(10).iloc[::-1]
ax.barh(top10['ID_CLIENTE'], top10['PROBABILIDADE_DETRACAO'], color='#176b87')
ax.set_xlim(0, 1)
ax.set_xlabel('Probabilidade estimada de detracao')
ax.set_title(f'Primeiros contatos sugeridos em {DIA_DEMO} | FILA SIMULADA')
fig.tight_layout()
plt.show()


In [ ]:
assert len(fila) == len(entrada_demo)
assert len(contatos) == 50 and not contatos.duplicated(['DIA_OPERACIONAL', 'ID_CLIENTE']).any()
assert fila['POSICAO_FILA'].notna().sum() == 60
fila_embaralhada, _ = pontuar_fila(entrada_demo.sample(frac=1, random_state=42), capacidade=50, t_score=T_SCORE, fuso=FUSO)
pd.testing.assert_frame_equal(fila, fila_embaralhada)
fila_zero, _ = pontuar_fila(entrada_demo, capacidade=0, t_score=T_SCORE, fuso=FUSO)
assert not fila_zero['PRIORIZADO'].any()
print('Verificado: contato unico, capacidade diaria, ordem deterministica e capacidade zero.')

EXPORTAR_FILA_SINTETICA = False  # Opcional; nenhum output e necessario para os graficos.
if EXPORTAR_FILA_SINTETICA:
    destino = RAIZ / 'outputs/fila-demonstracao-sintetica.csv'
    subprocess.run(['git', 'check-ignore', '--no-index', '-q', str(destino)], cwd=RAIZ, check=True)
    destino.parent.mkdir(parents=True, exist_ok=True)
    fila.to_csv(destino, index=False)
    print('Fila artificial exportada para diretorio local ignorado pelo Git.')
